In [5]:
system_prompt = """
You are a personal chef assistant.

Your role is to:
- Suggest recipes using the ingredients provided by the user.
- Minimize food waste.
- Respect the user's preferences stored in memory.
- Use the RAG tool to search recipes.
- Use the web search tool if the RAG does not contain enough information.
- Explain the preparation steps clearly.
"""

In [6]:
from langgraph.checkpoint.memory import InMemorySaver

In [8]:
from typing import Dict, Any
from tavily import TavilyClient
from dotenv import load_dotenv
from langchain.tools import tool
load_dotenv()
#Outil Web
tavily_client = TavilyClient()
@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for recent information."""
    return tavily_client.search(query)

In [10]:
#construction du RAG
from langchain_community.document_loaders import PyPDFLoader

loader1 = PyPDFLoader("Basic Cooking Recipes.pdf")
loader2 = PyPDFLoader("Cooking Tips.pdf")
loader3 = PyPDFLoader("ingredient_pairs.pdf")

documents = (
    loader1.load()
    + loader2.load()
    + loader3.load()
)

print(f"Nombre de documents : {len(documents)}")


Nombre de documents : 3


In [11]:
#découpage des documents en morceaux
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Nombre de chunks : {len(chunks)}")

Nombre de chunks : 3


In [12]:
#Création des embeddings
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2302.16it/s]


In [13]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

In [14]:
vector_store.add_documents(chunks)

['afcfe69b-04cb-46fd-b264-5125cf5b8a85',
 'c1c18622-7287-431c-8afd-d828e940eeb1',
 '863bc5a6-0539-4739-9737-26b3223d1e92']

In [15]:
results = vector_store.similarity_search(
    "recipe with zucchini"
)

print(results[0].page_content)

Basic Cooking Recipes 
 
Recipe : Cheese Omelette 
Ingredients: 
- Eggs 
- Cheese 
- Salt 
- Pepper 
- Butter 
Instructions: 
Beat the eggs, add cheese, then cook in a pan with butter for about 5 minutes. 
 
Recipe : Tomato Pasta 
Ingredients: 
- Pasta 
- Tomatoes 
- Garlic 
- Olive oil 
Instructions: 
Cook the pasta. Prepare a tomato sauce with garlic and olive oil. Mix together and serve. 
 
Recipe : Vegetable Soup 
Ingredients: 
- Carrot 
- Potato 
- Onion 
- Zucchini 
Instructions: 
Cut the vegetables into small pieces. Boil them for 30 minutes. Blend before serving.


In [16]:
#AGENT RAG
from langchain_core.tools import tool
@tool
def search_recipe(query: str) -> str:
    """Search recipes related to the user's ingredients."""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [18]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain_ollama import ChatOllama

# Initialiser le modèle
model = ChatOllama(
    model="llama3.2:3b",
    temperature=0
)

# Créer l'agent
agent = create_agent(
    model=model,
    tools=[search_recipe, web_search],
    checkpointer=InMemorySaver(),
    system_prompt=system_prompt,
)

# Configuration de la mémoire
config = {
    "configurable": {
        "thread_id": "1"
    }
}

In [20]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I am vegetarian."
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

Based on your preference for vegetarian recipes, I've searched for some delicious options using the RAG tool. Here are a few ideas:

1. **Vegetarian Pasta Primavera**: A colorful and flavorful spring-inspired pasta dish loaded with sautéed vegetables, garlic, and herbs, served with a side of garlic bread.
2. **Roasted Vegetable Quinoa Bowl**: A hearty and nutritious bowl filled with roasted vegetables such as sweet potatoes, Brussels sprouts, and cauliflower, served over quinoa with a dollop of tzatziki sauce.
3. **Grilled Portobello Mushroom Burgers**: Juicy portobello mushroom burgers grilled to perfection and served on a toasted bun with your favorite toppings, such as avocado, lettuce, and tomato.

These recipes are all vegetarian-friendly and can be made using common ingredients. Let me know if you'd like more ideas or have any specific preferences (e.g., gluten-free, dairy-free)!

Would you like me to provide the recipe for one of these options?


In [21]:
# Envoyer une requête
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="I have eggs and cheese, what can I cook?"
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

With eggs and cheese, you can make a delicious **Cheese Omelette**! Here's a simple recipe to get you started:

Ingredients:
- 2 eggs
- 1-2 slices of cheese (depending on your preference)
- Salt and pepper to taste
- 1 tablespoon of butter

Instructions:

1. Crack the eggs into a bowl and whisk them together with a pinch of salt and pepper.
2. Add the shredded cheese to the eggs and mix well.
3. Heat a non-stick pan over medium heat and add the butter. Once melted, pour in the egg mixture.
4. Let the eggs cook for about 2-3 minutes, until the edges start to set.
5. Use a spatula to gently lift and fold the edges of the omelette towards the center.
6. Continue cooking for another minute or until the cheese is melted and the eggs are cooked through.
7. Slide the omelette out of the pan onto a plate and serve hot.

Enjoy your cheesy egg delight!

Would you like any variations on this recipe or have any other questions?


In [24]:
# Envoyer une requête
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="How do I make Japanese ramen?"
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

To make Japanese ramen, you can follow a few different recipes and techniques. Here's a general guide to get you started:

**Ingredients:**

* Ramen noodles
* Pork bone broth (or chicken broth as a substitute)
* Tare (seasoning sauce)
* Chashu (braised pork belly)
* Nitamago (marinated eggs)
* Green onions, bean sprouts, and pickled ginger for garnish

**Broth:**

* To make the pork bone broth, simmer pork bones in water for several hours to extract the collagen and gelatin. Strain the broth and discard the solids.
* For a quicker option, use store-bought ramen broth or dashi powder.

**Tare:**

* Mix soy sauce, sake, mirin, and sugar to create a tare seasoning sauce.

**Chashu:**

* Braise pork belly in a mixture of soy sauce, sake, and sugar until tender and caramelized.

**Nitamago:**

* Marinate eggs in a mixture of soy sauce, sake, and sugar for several hours or overnight.

**Assembly:**

* Cook ramen noodles according to package instructions.
* Ladle the hot broth over the noodle